# Practical Exam: Spectrum Shades LLC
Spectrum Shades LLC is a prominent supplier of concrete color solutions, offering a wide range of pigments and coloring systems used in various concrete applications, including decorative concrete, precast concrete, and concrete pavers. The company prides itself on delivering high-quality colorants that meet the unique needs of its diverse clientele, including contractors, architects, and construction companies.
</br></br>
The company has recently observed a growing number of customer complaints regarding inconsistent color quality in their products. The discrepancies have led to a decline in customer satisfaction and a potential increase in product returns.
By identifying and mitigating the factors causing color variations, the company can enhance product reliability, reduce customer complaints, and minimize return rates.
</br></br>
You are part of the data analysis team tasked with providing actionable insights to help Spectrum Shades LLC address the issues of inconsistent color quality and improve customer satisfaction.

## Import packages & modules

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import colorcet as cc
import statistics
import datetime
import math
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import re
import random
import itertools
import missingno as msno
from scipy import stats
from collections import Counter
from matplotlib.lines import Line2D
from datetime import date, timedelta, datetime
from matplotlib.patches import Ellipse
from matplotlib.text import OffsetFrom
from fuzzywuzzy import process


from sklearn import linear_model
from sklearn import model_selection
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
#import statsmodels.formula.api from smf

#get a color palette
#seaborn_palette = sns.color_palette()

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `production_data` table, and the data team have provided the following data description. 

Write a query to return data matching this description. You must match all column names and description criteria.
</br>
Create a cleaned version of the dataframe.

- You should start with the data in the file "production_data.csv".
- Your output should be a dataframe named clean_data.
- All column names and values should match the table below.
</br>

| Column Name             | Criteria                                                                                         |
|--------------------------|--------------------------------------------------------------------------------------------------|
| batch_id | Discrete. Identifier for each batch. Missing values are not possible. |
| production_date | Date. Date when the batch was produced.|
| raw_material_supplier | Categorical. Supplier of the raw materials. (1='national_supplier', 2='international_supplier'). <br> Missing values should be replaced with 'national_supplier'.|
| pigment_type           | Nominal. Type of pigment used. ['type_a', 'type_b', 'type_c']. <br> Missing values should be replaced with 'other'. |
| pigment_quantity       | Continuous. Amount of pigment added (in kilograms) (Range: 1 - 100). <br> Missing values should be replaced with median. |
| mixing_time           | Continuous. Duration of the mixing process (in minutes). <br> Missing values should be replaced with mean. |
| mixing_speed          | Categorical. Speed of the mixing process represented as categories: 'Low', 'Medium', 'High'.</br> Missing values should be replaced with 'Not Specified'. |
| product_quality_score | Continuous. Overall quality score of the final product (rating on a scale of 1 to 10). <br> Missing values should be replaced with mean. |


In [40]:
#Task 1 -- MAKE SURE TO PUT ALL CODE INTO A SINGLE CELL
import pandas as pd
import numpy as np

#import data
    #include 'parse_dates' parameter to standardize the 'production_date' column with datetime data type values
orig_data = pd.read_csv('production_data.csv', parse_dates=['production_date'])
display(orig_data.sample(10))
display(orig_data.info())
display("Total # Missing values: {}".format(orig_data.isna().sum().sum()))

#search data for duplicates --> ARE NO COMPLETE DUPLICATES
    #display(orig_data[orig_data.duplicated()])

#inspect values of each column
#display(orig_data.product_quality_score.value_counts(dropna=False).sort_index())
#display(orig_data.product_quality_score.value_counts(dropna=False).sort_index().sum())
    #GOOD:         batch_id, production_date, pigment_quantity, product_quality_score
    #NEED TO FIX:  raw_material_supplier, pigment_type, mixing_time, mixing_speed



,batch_id,production_date,raw_material_supplier,pigment_type,pigment_quantity,mixing_time,mixing_speed,product_quality_score
181,182,2024-02-11,1,type_a,44.219599,19.78,Medium,7.752714
328,329,2024-06-11,1,type_b,37.243875,47.01,Low,7.869423
857,858,2024-01-07,2,type_a,37.065934,12.54,Medium,5.794634
1427,1428,2024-05-17,2,type_c,36.050205,13.40,Medium,5.243222
1977,1978,2024-04-29,2,type_c,37.169706,45.22,-,5.427325
1036,1037,2023-08-19,2,type_a,29.985063,13.00,Medium,5.822233
1463,1464,2024-05-12,2,type_b,36.058378,45.17,High,5.938616
1877,1878,2024-01-29,1,type_a,42.423510,8.31,Low,7.657551
680,681,2023-10-04,2,type_a,38.433412,35.89,Medium,7.286540
104,105,2023-09-03,2,Type_C,30.054780,10.17,Low,7.423219


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   batch_id               2000 non-null   int64         
 1   production_date        2000 non-null   datetime64[ns]
 2   raw_material_supplier  2000 non-null   int64         
 3   pigment_type           2000 non-null   object        
 4   pigment_quantity       2000 non-null   float64       
 5   mixing_time            1982 non-null   float64       
 6   mixing_speed           2000 non-null   object        
 7   product_quality_score  2000 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(2)
memory usage: 125.1+ KB


None

'Total # Missing values: 18'

In [55]:
#make a copy of original dataset to avoid modifying original
data = orig_data.copy()

display("INITIALLY:")
display(orig_data.mixing_speed.value_counts(dropna=False).sort_index())
display(orig_data.mixing_speed.value_counts(dropna=False).sort_index().sum())
#Fix values in columns as necessary
    #raw_material_supplier -- need to change data type from int to category, & change values
data = data.replace({'raw_material_supplier':{1:'national_supplier', 2:'international_supplier'}})
data['raw_material_supplier'] = data['raw_material_supplier'].astype('category')
    #pigment_type -- some values are incorrectly formatted: "Type_C" --> "type_c";
        #also, change data type from object to (nominal) category
data['pigment_type'] = [x.lower() for x in data.pigment_type]
data['pigment_type'] = data['pigment_type'].astype('category')
    #mixing_time -- impute missing values with mean (~35.28), round to 2 decimals to keep consistency with other non-missing values
data = data.fillna({'mixing_time':np.round(data.mixing_time.mean(),2)})
    #mixing_speed -- impute missing values w/"Not Specified", & change data type from object to category
data = data.replace({'mixing_speed':{'-':'Not Specified'}})
data['mixing_speed'] = data['mixing_speed'].astype('category')

display("----------------------", "AFTER:")
display(data.info())
display(data.mixing_speed.value_counts(dropna=False).sort_index())
display(data.mixing_speed.value_counts(dropna=False).sort_index().sum())


'INITIALLY:'

mixing_speed
-          34
High      650
Low       638
Medium    678
Name: count, dtype: int64

2000

'----------------------'

'AFTER:'

,batch_id,production_date,raw_material_supplier,pigment_type,pigment_quantity,mixing_time,mixing_speed,product_quality_score
1453,1454,2024-02-28,national_supplier,type_b,49.222950,53.97,Low,8.874857
149,150,2024-05-14,national_supplier,type_a,37.755788,31.67,Medium,6.703876
1774,1775,2024-01-31,national_supplier,type_c,43.154240,22.20,High,8.990386
549,550,2024-06-13,national_supplier,type_c,41.660911,39.82,Medium,8.318302
1476,1477,2024-07-25,international_supplier,type_a,36.214690,54.00,Low,6.776588
1258,1259,2024-01-08,national_supplier,type_a,42.554061,21.88,High,8.845043
513,514,2024-07-19,international_supplier,type_c,36.640689,57.17,High,6.678393
1108,1109,2024-01-13,international_supplier,type_b,29.972805,28.47,High,5.145911
1621,1622,2024-04-28,international_supplier,type_c,30.858174,35.28,Low,5.593201
1245,1246,2024-07-08,international_supplier,type_c,36.720733,42.92,Medium,5.631454


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   batch_id               2000 non-null   int64         
 1   production_date        2000 non-null   datetime64[ns]
 2   raw_material_supplier  2000 non-null   category      
 3   pigment_type           2000 non-null   category      
 4   pigment_quantity       2000 non-null   float64       
 5   mixing_time            2000 non-null   float64       
 6   mixing_speed           2000 non-null   category      
 7   product_quality_score  2000 non-null   float64       
dtypes: category(3), datetime64[ns](1), float64(3), int64(1)
memory usage: 84.6 KB


None

mixing_speed
High             650
Low              638
Medium           678
Not Specified     34
Name: count, dtype: int64

2000

In [72]:
#finalize & check cleaned data
clean_data = data.copy()
display(clean_data.info())
display("Total missing values: {}".format(clean_data.isna().sum().sum()))

display("BEFORE:")
display(orig_data.pigment_quantity.value_counts(dropna=False).sort_index())
display(orig_data.pigment_quantity.value_counts(dropna=False).sort_index().sum())

display("----------------", "AFTER:")
display(clean_data.pigment_quantity.value_counts(dropna=False).sort_index())
display(clean_data.pigment_quantity.value_counts(dropna=False).sort_index().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   batch_id               2000 non-null   int64         
 1   production_date        2000 non-null   datetime64[ns]
 2   raw_material_supplier  2000 non-null   category      
 3   pigment_type           2000 non-null   category      
 4   pigment_quantity       2000 non-null   float64       
 5   mixing_time            2000 non-null   float64       
 6   mixing_speed           2000 non-null   category      
 7   product_quality_score  2000 non-null   float64       
dtypes: category(3), datetime64[ns](1), float64(3), int64(1)
memory usage: 84.6 KB


None

'Total missing values: 0'

'BEFORE:'

pigment_quantity
19.413042    1
20.102625    1
20.252409    1
20.320349    1
20.749258    1
            ..
57.231083    1
57.655131    1
57.730527    1
57.847656    1
58.727253    1
Name: count, Length: 2000, dtype: int64

2000

'----------------'

'AFTER:'

pigment_quantity
19.413042    1
20.102625    1
20.252409    1
20.320349    1
20.749258    1
            ..
57.231083    1
57.655131    1
57.730527    1
57.847656    1
58.727253    1
Name: count, Length: 2000, dtype: int64

2000

# Task 2

You want to understand how the supplier type and quantity of materials affect the final product attributes.

Calculate the average `product_quality_score` and `pigment_quantity` grouped by `raw_material_supplier`.

- You should start with the data in the file 'production_data.csv'. 
- Your output should be a data frame named aggregated_data.
- It should include the three columns: `raw_material_supplier`, `avg_product_quality_score`, and `avg_pigment_quantity`.
- Your answers should be rounded to 2 decimal places.

In [83]:
#Task 2 -- MAKE SURE TO PUT ALL CODE INTO A SINGLE CELL

#Using original data ("orig_data"), get avg 'product_quality_score' & avg 'product_quality_score' per 'raw_material_supplier'
task2_data = orig_data.groupby('raw_material_supplier')[['product_quality_score','pigment_quantity']].mean().reset_index().rename(
    columns={'product_quality_score':'avg_product_quality_score', 'pigment_quantity':'avg_pigment_quantity'})
    #display(task2_data)
#Round values to 2 decimals
task2_data = np.round(task2_data, 2)
    #display(task2_data)
#Finalize data
aggregated_data = task2_data.copy()
display(aggregated_data)



,raw_material_supplier,avg_product_quality_score,avg_pigment_quantity
0,1,8.02,44.73
1,2,5.97,34.91


# Task 3

Identify all `product_quality_score` values for batches with a `raw_material_supplier` of 2 and a `pigment_quantity` greater than 35 kg. Use the original production data table, not the output of Task 2.
- You should start with the data in the file 'production_data.csv'.
- Your output should be a data frame named pigment_data.
- It should include the three columns: `raw_material_supplier`, `pigment_quantity`, and `product_quality_score`.
- Your answers should be rounded to 3 decimal places.

In [94]:
#Task 3 -- MAKE SURE TO PUT ALL CODE INTO A SINGLE CELL

#Using original data ("orig_data"), obtain all 'product_quality_score' values where 'raw_material_supplier' = 2 & 
    #'pigment_quantity' > 35 kg
    #There are 619 total such data points ('batch_id's)
task3_data = orig_data[(orig_data.raw_material_supplier == 2) & (orig_data.pigment_quantity > 35)][[
    'raw_material_supplier','pigment_quantity','product_quality_score']]
    #display(task3_data)
#Round values to 3 decimals
task3_data = np.round(task3_data, 3)
    #display(task3_data)
#Finalize data
pigment_data = task3_data.copy()
display(pigment_data)


,raw_material_supplier,pigment_quantity,product_quality_score
1,2,42.873,6.849
4,2,36.205,7.095
6,2,35.941,5.736
7,2,40.497,5.511
8,2,36.015,4.960
...,...,...,...
1985,2,40.484,7.027
1986,2,41.978,6.735
1994,2,37.484,4.501
1995,2,42.917,5.044


# Task 4

In order to proceed with further analysis later, you need to analyze how various factors relate to product quality. Start by calculating the mean and standard deviation for the following columns: `pigment_quantity`, and `product_quality_score`. </br> These statistics will help in understanding the central tendency and variability of the data related to product quality.
</br> </br >
Next, calculate the Pearson correlation coefficient between the following variables: `pigment_quantity`, and `product_quality_score`.
</br>
These correlation coefficients will provide insights into the strength and direction of the relationships between the factors and overall product quality.


- You should start with the data in the file 'production_data.csv'.
- Calculate the mean and standard deviation for the columns pigment_quantity and product_quality_score as: `product_quality_score_mean`, `product_quality_score_sd`, `pigment_quantity_mean`, `pigment_quantity_sd`.
- Calculate the Pearson correlation coefficient between pigment_quantity and product_quality_score as: `corr_coef`
- Your output should be a data frame named product_quality.
- It should include the columns: `product_quality_score_mean`, `product_quality_score_sd`, `pigment_quantity_mean`, `pigment_quantity_sd`, `corr_coef`.
- Ensure that your answers are rounded to 2 decimal places.


In [108]:
#Task 4 -- MAKE SURE TO PUT ALL CODE INTO A SINGLE CELL

#Using original data, calculate avg & standard deviation of 'pigment_quantity' & 'product_quality_score'
task4_dataI = orig_data.agg({'product_quality_score':['mean','std'], 'pigment_quantity':['mean','std']})
display(task4_dataI)
#Calculate correlation (Pearson) between 'pigment_quantity' & 'product_quality_score'
task4_dataII = orig_data[['pigment_quantity','product_quality_score']].corr(method='pearson')
display(task4_dataII)

#Compile desired data into a single dataframe
task4_dict = {'product_quality_score_mean':task4_dataI.iloc[0,0], 'product_quality_score_sd':task4_dataI.iloc[1,0], 
             'pigment_quantity_mean':task4_dataI.iloc[0,1], 'pigment_quantity_sd':task4_dataI.iloc[1,1], 
             'corr_coef':task4_dataII.iloc[0,1]}
product_quality = pd.DataFrame.from_dict(task4_dict, orient='index').T
#Round values to 2 decimals
product_quality = np.round(product_quality, 2)
display(product_quality)


,product_quality_score,pigment_quantity
mean,6.684824,38.346191
std,1.387717,6.826539


,pigment_quantity,product_quality_score
pigment_quantity,1.000000,0.485482
product_quality_score,0.485482,1.000000


,product_quality_score_mean,product_quality_score_sd,pigment_quantity_mean,pigment_quantity_sd,corr_coef
0,6.68,1.39,38.35,6.83,0.49
